In [43]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week6-lesson-3"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

#### Drop column

In [2]:
# /public/trendytech/orders_wh

In [9]:
df_orders = spark.read \
.format("csv") \
.option("header","true") \
.option("inferSchema","true")\
.load("/public/trendytech/orders_wh")

In [10]:
df_orders.show(5)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
|       4|2013-07-25 00:00:...|       8827|         CLOSED|
|       5|2013-07-25 00:00:...|      11318|       COMPLETE|
+--------+--------------------+-----------+---------------+
only showing top 5 rows



In [11]:
df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
# /public/trendytech/retail_db/order_items/part-0000
# order_item_id,order_id,product_id,quantity,subtotal,product_price

In [13]:
raw_df = spark.read \
.format("csv") \
.option("inferSchema","true")\
.load("/public/trendytech/retail_db/order_items/part-00000")

In [14]:
raw_df.show(4)

+---+---+----+---+------+------+
|_c0|_c1| _c2|_c3|   _c4|   _c5|
+---+---+----+---+------+------+
|  1|  1| 957|  1|299.98|299.98|
|  2|  2|1073|  1|199.99|199.99|
|  3|  2| 502|  5| 250.0|  50.0|
|  4|  2| 403|  1|129.99|129.99|
+---+---+----+---+------+------+
only showing top 4 rows



In [15]:
raw_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: double (nullable = true)
 |-- _c5: double (nullable = true)



In [21]:
cols = ["order_item_id","order_id","product_id","quantity","subtotal","product_price"]

In [25]:
newdf = raw_df.toDF(*cols)

In [27]:
# or newdf = raw_df.toDF("order_item_id","order_id","product_id","quantity","subtotal","product_price")

In [26]:
newdf.show(3)

+-------------+--------+----------+--------+--------+-------------+
|order_item_id|order_id|product_id|quantity|subtotal|product_price|
+-------------+--------+----------+--------+--------+-------------+
|            1|       1|       957|       1|  299.98|       299.98|
|            2|       2|      1073|       1|  199.99|       199.99|
|            3|       2|       502|       5|   250.0|         50.0|
+-------------+--------+----------+--------+--------+-------------+
only showing top 3 rows



In [28]:
df1 = newdf.drop("subtotal")

In [29]:
df1.show(3)

+-------------+--------+----------+--------+-------------+
|order_item_id|order_id|product_id|quantity|product_price|
+-------------+--------+----------+--------+-------------+
|            1|       1|       957|       1|       299.98|
|            2|       2|      1073|       1|       199.99|
|            3|       2|       502|       5|         50.0|
+-------------+--------+----------+--------+-------------+
only showing top 3 rows



 ### expr()

In [33]:
df2 = df1.select("*",expr("quantity * product_price as subtotal"))

In [34]:
df2.printSchema()

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- product_price: double (nullable = true)
 |-- subtotal: double (nullable = true)



In [35]:
df2.show(3)

+-------------+--------+----------+--------+-------------+--------+
|order_item_id|order_id|product_id|quantity|product_price|subtotal|
+-------------+--------+----------+--------+-------------+--------+
|            1|       1|       957|       1|       299.98|  299.98|
|            2|       2|      1073|       1|       199.99|  199.99|
|            3|       2|       502|       5|         50.0|   250.0|
+-------------+--------+----------+--------+-------------+--------+
only showing top 3 rows



 ### selectExpr

In [36]:
df3 = df1.selectExpr("*","quantity * product_price as subtotal")

In [37]:
df3.show(3)

+-------------+--------+----------+--------+-------------+--------+
|order_item_id|order_id|product_id|quantity|product_price|subtotal|
+-------------+--------+----------+--------+-------------+--------+
|            1|       1|       957|       1|       299.98|  299.98|
|            2|       2|      1073|       1|       199.99|  199.99|
|            3|       2|       502|       5|         50.0|   250.0|
+-------------+--------+----------+--------+-------------+--------+
only showing top 3 rows



In [38]:
# [itv024128@g02 ~]$ hadoop fs -head  /public/trendytech/retail_db/products/part-00000
# 1,2,Quest Q64 10 FT. x 10 FT. Slant Leg Instant U,,59.98,http://images.acmesports.sports/Quest+Q64+10+FT.+x+10+FT.+Slant+Leg+Instant+Up+Canopy
# 2,2,Under Armour Men's Highlight MC Football Clea,,129.99,http://images.acmesports.sports/Under+Armour+Men%27s+Highlight+MC+Football+Cleat3

### withColumn() with expr()

In [44]:
prod_df = spark.read.format("csv").option("inferSchema","true").load("/public/trendytech/retail_db/products/part-00000")

In [45]:
prod_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: double (nullable = true)
 |-- _c5: string (nullable = true)



In [46]:
prod_df1 = prod_df.toDF("prod_id","cat_id","name","desc","price","url")

In [54]:
prod_df1.show(5)

+-------+------+--------------------+----+------+--------------------+
|prod_id|cat_id|                name|desc| price|                 url|
+-------+------+--------------------+----+------+--------------------+
|      1|     2|Quest Q64 10 FT. ...|null| 59.98|http://images.acm...|
|      2|     2|Under Armour Men'...|null|129.99|http://images.acm...|
|      3|     2|Under Armour Men'...|null| 89.99|http://images.acm...|
|      4|     2|Under Armour Men'...|null| 89.99|http://images.acm...|
|      5|     2|Riddell Youth Rev...|null|199.99|http://images.acm...|
+-------+------+--------------------+----+------+--------------------+
only showing top 5 rows



In [48]:
prod_df2 = prod_df1.withColumn("price",expr("price * 1.2"))

In [55]:
prod_df2.show(5)

+-------+------+--------------------+----+------------------+--------------------+
|prod_id|cat_id|                name|desc|             price|                 url|
+-------+------+--------------------+----+------------------+--------------------+
|      1|     2|Quest Q64 10 FT. ...|null|            71.976|http://images.acm...|
|      2|     2|Under Armour Men'...|null|           155.988|http://images.acm...|
|      3|     2|Under Armour Men'...|null|107.98799999999999|http://images.acm...|
|      4|     2|Under Armour Men'...|null|107.98799999999999|http://images.acm...|
|      5|     2|Riddell Youth Rev...|null|           239.988|http://images.acm...|
+-------+------+--------------------+----+------------------+--------------------+
only showing top 5 rows



### CASE WHEN THEN END inside expr()

In [50]:
prod_df3 = prod_df1.withColumn("price",expr("CASE  WHEN name like '%Quest%'  THEN  price * 1.2   WHEN name like '%Under%'  THEN  price * 1.1  ELSE price END"))

In [53]:
prod_df3.show(5)

+-------+------+--------------------+----+------------------+--------------------+
|prod_id|cat_id|                name|desc|             price|                 url|
+-------+------+--------------------+----+------------------+--------------------+
|      1|     2|Quest Q64 10 FT. ...|null|            71.976|http://images.acm...|
|      2|     2|Under Armour Men'...|null|142.98900000000003|http://images.acm...|
|      3|     2|Under Armour Men'...|null|            98.989|http://images.acm...|
|      4|     2|Under Armour Men'...|null|            98.989|http://images.acm...|
|      5|     2|Riddell Youth Rev...|null|            199.99|http://images.acm...|
+-------+------+--------------------+----+------------------+--------------------+
only showing top 5 rows



### CASE WHEN THEN END inside selectExpr()

In [60]:
prod_df4 = prod_df1.selectExpr("prod_id","cat_id","name","desc", "CASE WHEN name like '%Quest%'  THEN  price * 1.2   WHEN name like '%Under%'  THEN  price * 1.1  ELSE price END as price","url")

In [61]:
prod_df4.show(5)

+-------+------+--------------------+----+------------------+--------------------+
|prod_id|cat_id|                name|desc|             price|                 url|
+-------+------+--------------------+----+------------------+--------------------+
|      1|     2|Quest Q64 10 FT. ...|null|            71.976|http://images.acm...|
|      2|     2|Under Armour Men'...|null|142.98900000000003|http://images.acm...|
|      3|     2|Under Armour Men'...|null|            98.989|http://images.acm...|
|      4|     2|Under Armour Men'...|null|            98.989|http://images.acm...|
|      5|     2|Riddell Youth Rev...|null|            199.99|http://images.acm...|
+-------+------+--------------------+----+------------------+--------------------+
only showing top 5 rows



### Remove dups

In [ ]:
# distinct()

In [63]:
mylist = [
    (1,"A",34),
    (1,"A",34),
    (2,"B",34),
    (3,"B",34),
    (4,"D",36),
]

In [66]:
dfdup = spark.createDataFrame(mylist,("id","name","age"))

In [67]:
dfdup.show()

+---+----+---+
| id|name|age|
+---+----+---+
|  1|   A| 34|
|  1|   A| 34|
|  2|   B| 34|
|  3|   B| 34|
|  4|   D| 36|
+---+----+---+



In [68]:
dfdup.distinct().show()

+---+----+---+
| id|name|age|
+---+----+---+
|  1|   A| 34|
|  4|   D| 36|
|  2|   B| 34|
|  3|   B| 34|
+---+----+---+



In [69]:
dfdup.dropDuplicates().show()

+---+----+---+
| id|name|age|
+---+----+---+
|  1|   A| 34|
|  4|   D| 36|
|  2|   B| 34|
|  3|   B| 34|
+---+----+---+



In [71]:
dfdup.dropDuplicates(["name"]).show()

+---+----+---+
| id|name|age|
+---+----+---+
|  2|   B| 34|
|  4|   D| 36|
|  1|   A| 34|
+---+----+---+



In [73]:
dfdup.dropDuplicates(["name","id"]).show()

+---+----+---+
| id|name|age|
+---+----+---+
|  4|   D| 36|
|  1|   A| 34|
|  3|   B| 34|
|  2|   B| 34|
+---+----+---+



In [74]:
dfdup.dropDuplicates(["name","id","age"]).show()

+---+----+---+
| id|name|age|
+---+----+---+
|  4|   D| 36|
|  2|   B| 34|
|  3|   B| 34|
|  1|   A| 34|
+---+----+---+

